Import libraries


In [19]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Tarnet/main_tarnet.ipynb")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [20]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 25.8 ms, sys: 6.02 ms, total: 31.8 ms
Wall time: 31.4 ms
CPU times: user 11.7 ms, sys: 6.01 ms, total: 17.7 ms
Wall time: 17.6 ms
CPU times: user 4.54 ms, sys: 1 ms, total: 5.54 ms
Wall time: 5.52 ms


In [21]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [22]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [23]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [24]:
print('y_train[:10]', y_train[:10].astype(float))

y_train[:10] [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [25]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [26]:
epochs = 70
lr = 5e-4
wd = 1e-5
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 15
shared_dropout = 0
outcome_dropout = 0
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")
print (f" activation = {activation}")

 epochs = 70
 learning rate = 0.0005
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 15
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30
 activation = <class 'torch.nn.modules.activation.ReLU'>


In [27]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence CFRNet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            tarnet.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p= tarnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 0. Best value: 0.73179:   2%|▏         | 1/50 [00:47<39:03, 47.82s/it]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 1. Best value: 0.736469:   4%|▍         | 2/50 [01:37<38:56, 48.67s/it]

...

Best trial: 34. Best value: 0.760246:  96%|█████████▌| 48/50 [44:51<02:01, 60.91s/it]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 34. Best value: 0.760246:  98%|█████████▊| 49/50 [45:54<01:01, 61.69s/it]

Locked random seed: 412312
Locked random seed: 42
Locked random seed: 1874
Locked random seed: 902745
Locked random seed: 1


Best trial: 34. Best value: 0.760246: 100%|██████████| 50/50 [46:44<00:00, 56.10s/it]


In [28]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 34
Best mean_Val_AUQC: 0.760246
Best params: {'lr': 0.00038056977654063104, 'weight_decay': 9.253713480199685e-05}

Best config table:


,lr,weight_decay,mean_Val_AUQC,std_Val_AUQC
0,0.000381,0.000093,0.760246,0.052533



Top 10 trials:


,trial,lr,weight_decay,mean_Val_AUQC,std_Val_AUQC
0,34,0.000381,0.000093,0.760246,0.052533
1,43,0.000454,0.000100,0.760232,0.044013
2,41,0.000383,0.000082,0.759706,0.054251
3,38,0.000382,0.000082,0.759437,0.054893
4,36,0.000459,0.000096,0.759366,0.043022
5,44,0.000457,0.000090,0.759313,0.042886
6,35,0.000384,0.000100,0.758375,0.052372
7,32,0.000384,0.000053,0.757791,0.054788
8,26,0.000453,0.000066,0.757363,0.044630
9,45,0.000467,0.000077,0.752927,0.045036


In [30]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1], epochs=150, learning_rate=0.000381, 
        weight_decay=0.000093, use_ema=True, ema_alpha=0.35, patience=20,
        shared_hidden=200, outcome_hidden=100,
        outcome_droupout=0., shared_dropout=0,
        early_stop_metric="qini", early_stop_start_epoch=30,
    )
    
    tarnet.fit(train_loader, val_loader)
    
    # --- Predict on VALIDATION set ---
    # Giả sử bạn có x_men_val_t, y_men_val_t, t_men_val_t từ val_loader hoặc tương đương
    x_val_device = x_men_val_t.to(device)
    y0_val_p, y1_val_p = tarnet.predict(x_val_device)
    uplift_val = (y1_val_p - y0_val_p).cpu().numpy().flatten()
    y_val_true_np = y_men_val_t.cpu().numpy().flatten()
    t_val_true_np = t_men_val_t.cpu().numpy().flatten()
    
    current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

    # --- Predict on TEST set ---
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán ATE
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'Val_AUQC': current_val_auqc, # Thêm Val AUQC vào đây
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED} | Val AUQC: {current_val_auqc:.4f}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

# Đưa cột Val_AUQC lên đầu cho dễ nhìn
cols = ['Seed', 'Val_AUQC', 'AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']
df_results = df_results[cols]

print("\n" + "="*95)
print(f"{'CHI TIẾT TỪNG SEED (VALIDATION & TEST)':^95}")
print("="*95)
print(df_results.to_string(index=False, formatters={
    'Val_AUQC': '{:,.4f}'.format, 'AUUC': '{:,.4f}'.format, 
    'AUQC': '{:,.4f}'.format, 'Lift': '{:,.4f}'.format, 
    'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*95)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^95}")
print("-" * 95)
for metric in ['Val_AUQC', 'AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*95)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.35)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 1.3711 | Val Loss: 1.3611 | Val Qini: 0.4363 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4363 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 1.3309 | Val Loss: 1.3200 | Val Qini: 0.2596 (patience: 1/20)EMA Trend: 0.3745 | (patience: 1/20)
Epoch 3/150 | Loss: 1.2795 | Val Loss: 1.2641 | Val Qini: 0.2419 (patience: 2/20)EMA Trend: 0.3281 | (patience: 2/20)
Epoch 4/150 | Loss: 1.2014 | Val Loss: 1.1743 | Val Qini: 0.2792 (patience: 3/20)EMA Trend: 0.3110 | (patience: 3/20)
Epoch 5/150 | Loss: 1.0624 | Val Loss: 1.0169 | Val Qini: 0.3136 ✓ above trend but not peak (patience: 4/20)EMA Trend: 0.3119 | ✓ above trend but not peak (patie